[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-monte-carlo-markov.ipynb)

# Monte Carlo Simulation & Markov Chains

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

Two techniques for reasoning about randomness without a closed-form formula: simulate the process directly, many times, and read the answer off the results.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **⚠ Why This Page Is Marked "Advanced"**
>
> Both techniques here trade a closed-form formula for repeated computation — instead of deriving an exact answer analytically, you write code that simulates the process thousands of times and reads the answer off the empirical results. This is a different way of thinking about probability than anything else in this course, and it underpins the Bayesian machinery (MCMC) covered on the next two pages.

## Monte Carlo Simulation — The Core Idea

You already met one simulation-based technique on the Bootstrap Resampling page: resampling *with replacement from your own observed data* to see how a statistic varies. Monte Carlo simulation asks a related but distinct question — instead of resampling the data you have, you simulate fresh data **from a fully-specified theoretical model** (a known distribution with known parameters), repeat that many times, and use the resulting spread to answer questions a formula can't easily answer.

| Bootstrap (already covered) | Monte Carlo Simulation (this page) |
|---|---|
| Resamples *your actual observed data*, with replacement | Simulates fresh data from a *known, fully-specified distribution* |
| Answers: "how would my statistic vary across different samples from whatever population my data came from?" | Answers: "if this theoretical model were exactly true, what would the statistic's distribution look like?" |
| Requires no assumption about the population's shape | Requires committing to a specific model (e.g. "assume the null hypothesis is exactly true") |

## Worked Example — Testing Normality via a Simulated Null Distribution

The Kolmogorov-Smirnov (KS) test checks whether a sample plausibly came from a reference distribution, using the maximum gap between the sample's empirical CDF and the reference CDF as its test statistic, D. Rather than using the KS test's known asymptotic formula for the p-value, we can build the null distribution of D ourselves by simulation — useful for tests or statistics where no such formula exists. This reuses the exact same 500-row Swiggy order-value dataset from the Bootstrap page:

In [ ]:
import numpy as np
from scipy.stats import norm

# Same 500-row Swiggy order-value dataset as the Bootstrap page (same seed)
np.random.seed(9)
orders = np.concatenate([np.random.normal(350,80,480), np.random.normal(2200,400,20)])
n = len(orders)

# Step 1: estimate parameters, standardize the data
xbar, s = orders.mean(), orders.std(ddof=1)
standardized = (orders - xbar) / s

# Step 2: observed KS statistic D_obs vs. the standard normal
sorted_std = np.sort(standardized)
ecdf = np.arange(1, n+1) / n
D_obs = np.max(np.abs(ecdf - norm.cdf(sorted_std)))
print(f"D_obs = {D_obs:.4f}")

# Step 3: Monte Carlo simulate the null distribution — 1000 fresh N(0,1) samples
num_sims = 1000
D_sims = np.zeros(num_sims)
for i in range(num_sims):
    sim = np.sort(np.random.normal(0, 1, n))
    D_sims[i] = np.max(np.abs(ecdf - norm.cdf(sim)))

# Step 4: p-value = fraction of simulated D's at least as extreme as D_obs
p_value = np.mean(D_sims >= D_obs)
print(f"Monte Carlo p-value = {p_value:.4f}")
print(f"Null distribution D: mean={D_sims.mean():.4f}, range=[{D_sims.min():.4f}, {D_sims.max():.4f}]")

Every one of the 1,000 simulated null-distribution D values tops out at 0.0794 — nowhere near the observed D_obs=0.3562. The Monte Carlo p-value of ≈0 correctly **rejects** the normality assumption, driven entirely by the 20 large bulk orders (mean ₹2,200) sitting in an otherwise ₹350-centred distribution. This is the same real dataset from the Bootstrap page — the fat right tail that widened the mean's bootstrap CI there is precisely what breaks the normality assumption here.

## Markov Chains — Modeling State-to-State Transitions

A Markov chain models a system that moves between a fixed set of **states**, where the probability of the next state depends *only* on the current state — not on the entire history that led there. This "memorylessness" is the Markov property:

$$P(X_{t+1}=j \mid X_t=i, X_{t-1},\dots,X_0) = P(X_{t+1}=j\mid X_t=i) \qquad \text{the transition matrix } P \text{ holds every } P(i\to j)$$

Each row of the transition matrix P must sum to 1 (from any given state, you go *somewhere*, with total probability 1). A state that transitions only to itself with probability 1 is called **absorbing** — once you enter it, you never leave.

## Worked Example — A Support Ticket's Journey

Model a helpdesk ticket's lifecycle as a 5-state Markov chain: **New → Assigned → InProgress ⇄ Escalated → Resolved**, with Resolved absorbing. Simulating 30,000 tickets and tracking every path taken:

In [ ]:
import numpy as np
from collections import Counter

states = ['New', 'Assigned', 'InProgress', 'Escalated', 'Resolved']
idx = {s: i for i, s in enumerate(states)}

# Rows sum to 1. Resolved (row 4) is absorbing.
P = np.array([
    [0.0, 1.0,  0.0,  0.0,  0.0],   # New       -> Assigned
    [0.0, 0.0,  0.85, 0.15, 0.0],   # Assigned  -> InProgress / Escalated
    [0.0, 0.0,  0.25, 0.15, 0.60],  # InProgress-> stays / Escalated / Resolved
    [0.0, 0.0,  0.35, 0.15, 0.50],  # Escalated -> InProgress / stays / Resolved
    [0.0, 0.0,  0.0,  0.0,  1.0],   # Resolved  -> absorbing
])

def simulate_path():
    current = idx['New']
    path = ['New']
    while current != idx['Resolved']:
        nxt = np.random.choice(5, p=P[current])
        path.append(states[nxt]); current = nxt
    return tuple(path)

paths = [simulate_path() for _ in range(30000)]
counts = Counter(paths)
for path, count in counts.most_common(3):
    print(f"{' -> '.join(path)}  ({count/300:.1f}%)")

lengths = [len(p)-1 for p in paths]
print(f"Average steps to resolution: {np.mean(lengths):.2f}")
print(f"Escalated at least once: {np.mean(['Escalated' in p for p in paths])*100:.1f}%")

Just over half of all tickets resolve via the direct 3-step path with no detours, but almost a third get escalated at least once before resolution — a number that would be tedious to derive analytically from the transition matrix directly (it requires summing infinitely many possible looping paths), but falls straight out of simulation.

## Try It — Simulate Tickets Through the Chain

This is the exact transition matrix from the code above. Click below to simulate one ticket at a time and watch it move through the chain, or run a batch of 1,000 to see the aggregate statistics converge toward the real 30,000-run numbers.

> **💡 When to Reach for Each Technique**
>
> Use **Monte Carlo simulation** when you need the distribution of a statistic under a specific, fully-specified model (a null hypothesis, a theoretical distribution) and no clean formula exists — or you'd rather not trust the formula's assumptions. Use a **Markov chain** whenever a system moves through a fixed set of states and the next-state probability depends only on the current state — funnels, lifecycle stages, queueing systems, and (as the next page shows) the sampling mechanism inside MCMC itself.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Estimate π by throwing darts

Draw 200,000 random points in the unit square. The share that lands inside the quarter circle (x² + y² < 1) times 4 estimates π. Store it in `pi_hat`.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
pi_hat = None   # TODO


In [ ]:
try:
    check("within 0.02 of pi", abs(pi_hat - np.pi) < 0.02)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
rng = np.random.default_rng(0)
xy = rng.random((200_000, 2))
pi_hat = 4 * np.mean((xy ** 2).sum(axis=1) < 1)

```

</details>

### Exercise 2 · Medium · The long-run behaviour of a Markov chain

A support desk is either `Quiet` or `Busy`, with transition matrix `P` (rows sum to 1). Multiply a starting distribution by `P` repeatedly (200 times) and store the limit in `stationary`. It should satisfy `stationary @ P == stationary`.

In [ ]:
import numpy as np
P = np.array([[0.9, 0.1], [0.5, 0.5]])
stationary = None   # TODO


In [ ]:
try:
    check("is a probability vector", abs(stationary.sum() - 1) < 1e-9)
    check("equals [5/6, 1/6]", np.allclose(stationary, [5 / 6, 1 / 6]))
    check("is stationary", np.allclose(stationary @ P, stationary))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
P = np.array([[0.9, 0.1], [0.5, 0.5]])
stationary = np.array([1.0, 0.0])
for _ in range(200):
    stationary = stationary @ P

```

</details>

### Exercise 3 · Stretch · Expected steps to absorption

A ticket moves `A → B` always; from `B` it is resolved with probability 0.7 or bounced back to `A` with probability 0.3. Using the transient block `Q` and the fundamental matrix `N = (I - Q)^-1`, store in `steps_from_A` the expected number of steps until resolution starting from `A`.

In [ ]:
import numpy as np
Q = np.array([[0.0, 1.0], [0.3, 0.0]])   # transient states A, B
steps_from_A = None   # TODO


In [ ]:
try:
    check("expected steps = 2/0.7", abs(steps_from_A - 2 / 0.7) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
Q = np.array([[0.0, 1.0], [0.3, 0.0]])
N = np.linalg.inv(np.eye(2) - Q)
steps_from_A = float(N[0].sum())

```

</details>

---
*Back to the course: **Machine Learning End To End → Monte Carlo Simulation & Markov Chains**.*